# 04 — Modeling: SBERT Embeddings & Semantic Search

Builds the actual search pipeline on top of the featured dataset: SBERT embeddings, Groq-based query enhancement, semantic similarity search, quality-aware re-ranking, and Steam API enrichment.

This is the logic that was later extracted into `nlp_model/` so the FastAPI backend could reuse it.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import os
import torch
from dotenv import load_dotenv

load_dotenv('../.env')

df = pd.read_pickle('../data/df_clean.pkl')
df.shape

## 2. Generating SBERT embeddings

`paraphrase-multilingual-mpnet-base-v2` was chosen so the same model understands both English and Spanish queries, without a separate translation step.

In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')

game_embeddings = model.encode(
    df['embedding'].tolist(),
    convert_to_tensor=True,
    show_progress_bar=True
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/476 [00:00<?, ?it/s]

## 3. Query enhancement with Groq (LLaMA 3)

Before searching, the user's query is translated to English if needed and its key terms are repeated to weight them more heavily in the embedding space. This was the prompt that gave the most consistent results after a few iterations.

In [11]:
from groq import Groq

groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

def mejorar_consulta(consulta):
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": f"""You are a video game expert.
        Given this search query: '{consulta}'

        1. Translate the query to English if it is in another language
        2. Write the translated query
        2. Extract the main theme/enemy/setting keywords
        3. Repeat those keywords 3 times to give them more weight
        4. Add 5 related gaming terms in English

        Example:
        'kill horde of demons' → 'demons demons demons, kill horde of demons,
        demon slayer, hellish, gore, FPS, shooter'

        Return only the words, no explanation."""}],
        max_tokens=200
    )
    return response.choices[0].message.content

consulta = "kill horde of demons, shooter"
consulta_mejorada = mejorar_consulta(consulta)

print("Original:", consulta)
print("Mejorada:", consulta_mejorada)

Original: kill horde of demons, shooter
Mejorada: kill horde of demons 
kill horde of demons, shooter 
demons demons demons, kill horde of demons, shooter, dark fantasy, hellish, gore, FPS, action


## 4. Semantic search

In [12]:
from sentence_transformers import util

query_embedding = model.encode(consulta_mejorada, convert_to_tensor=True)
cosine_scores = util.cos_sim(query_embedding, game_embeddings)[0]
top_results = torch.topk(cosine_scores, k=10)

for score, idx in zip(top_results.values, top_results.indices):
    game = df.iloc[idx.item()]
    print(f"{game['name']} | match: {round(score.item(), 4)}")

Painkiller Hell & Damnation | match: 0.7297
Killing Floor | match: 0.6852
25 Cadre of Death | match: 0.6775
Killbot | match: 0.6769
Slain: Back from Hell | match: 0.6721
DemonsAreCrazy | match: 0.6668
Feral Fury | match: 0.6655
Hell is Other Demons | match: 0.665
Devil Daggers | match: 0.6595
Killer is Dead - Nightmare Edition | match: 0.659


## 5. Quality-aware re-ranking

Raw semantic match alone would happily rank an obscure, poorly-reviewed game above a beloved one if the text happens to line up. Blending in `quality_score` fixes that.

In [13]:
resultados = []
for score, idx in zip(top_results.values, top_results.indices):
    game = df.iloc[idx.item()]
    quality = float(game['quality_score']) if pd.notna(game['quality_score']) else 0.0
    resultados.append({'name': game['name'], 'appid': game['appid'], 'match': round(score.item(), 4), 'quality_score': quality})

for r in resultados:
    r['score_final'] = r['match'] * 0.55 + r['quality_score'] * 0.45

resultados = sorted(resultados, key=lambda x: x['score_final'], reverse=True)

for r in resultados[:10]:
    print(f"{r['name']} | match: {r['match']} | quality: {r['quality_score']} | final: {round(r['score_final'], 4)}")

Killing Floor | match: 0.6852 | quality: 0.7787 | final: 0.7273
Devil Daggers | match: 0.6595 | quality: 0.7274 | final: 0.6901
Painkiller Hell & Damnation | match: 0.7297 | quality: 0.6077 | final: 0.6748
Killer is Dead - Nightmare Edition | match: 0.659 | quality: 0.6161 | final: 0.6397
Slain: Back from Hell | match: 0.6721 | quality: 0.5884 | final: 0.6344
Hell is Other Demons | match: 0.665 | quality: 0.5934 | final: 0.6328
Feral Fury | match: 0.6655 | quality: 0.482 | final: 0.5829
DemonsAreCrazy | match: 0.6668 | quality: 0.4096 | final: 0.5511
25 Cadre of Death | match: 0.6775 | quality: 0.3409 | final: 0.526
Killbot | match: 0.6769 | quality: 0.2775 | final: 0.4972


## 6. Steam API — live price & trailer

The last step enriches the top results with real-time price and trailer data from Steam's public API.

In [14]:
import requests

def get_steam_data(appid):
    url = f"https://store.steampowered.com/api/appdetails?appids={appid}&cc=pe"
    response = requests.get(url, timeout=10)
    data = response.json()

    if data[str(appid)]['success']:
        game = data[str(appid)]['data']
        price_overview = game.get('price_overview', {})
        return {
            'price': price_overview.get('final_formatted'),
            'original_price': price_overview.get('initial_formatted'),
            'discount': price_overview.get('discount_percent'),
            'trailer': game.get('movies', [{}])[0].get('hls_h264') if game.get('movies') else None
        }
    return {}

top_game = resultados[0]
get_steam_data(top_game['appid'])

{'price': 'S/.49.95',
 'original_price': '',
 'discount': 0,
 'trailer': 'https://video.akamai.steamstatic.com/store_trailers/1250/1890/bff2359d4493fd434a4b35f685c1b510bcb4529a/1750483898/hls_264_master.m3u8?t=1529340856'}

---

This pipeline was later split into `nlp_model/embeddings.py`, `llm.py`, `scoring.py` and `steam.py` so the FastAPI backend could reuse it directly.